# Projeto Final IF1014 — Mineração de Dados (CRISP-DM)

**Dataset:** Diabetes 130-US Hospitals (UCI 296) — classificação multiclasse de readmissão.

**Métrica principal:** F1-macro (dataset desbalanceado; classe `<30` é minoritária).

Este notebook executa o workflow obrigatório de ponta a ponta. As seções seguem a ordem exigida pelo `docs/exigencias.pdf` (Seção 7). **O teste é tocado APENAS na Seção 9**, via `load_test(unlock_token='I_AM_IN_FINAL_EVALUATION')`.

Reprodução completa:
```bash
uv sync
uv run jupyter nbconvert --to notebook --execute main.ipynb --output main.executed.ipynb
```


## 1. Setup e imports

In [ ]:
from __future__ import annotations

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader import load_train, split_info
from src.evaluation import (
    format_summary,
    paired_compare,
    summary_from_search,
)
from src.models import checkpoint_models
from src.preprocessing import build_baseline_pipeline
from src.search import best_per_fold_scores, run_search
from src.utils import RANDOM_STATE, ensure_dirs, set_global_seed

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 4.5)

set_global_seed(RANDOM_STATE)
ensure_dirs()
print(f'Seed global = {RANDOM_STATE}')


## 2. Carga bruta + split estratificado

O split é materializado uma única vez por `src/data_loader.py` (estratificado, `random_state=42`, `test_size=0.2`) e salvo em `data/interim/`. A partir daqui o teste fica isolado por trás da guarda do token.

**O briefing do cliente fictício** será preenchido no relatório (template em `docs/exigencias.pdf` Seção 2.2). Resumo do cenário: hospital quer prever readmissão precoce (<30 dias) para acionar acompanhamento pós-alta; falso negativo na classe `<30` tem custo alto.

In [ ]:
X_train, y_train = load_train()
info = split_info()
print('Treino:', info['train_shape'])
print('Teste (apenas contagem):', info['test_n_rows'], 'linhas')
print('\nBalanço de classes no TREINO:')
for cls, frac in info['class_balance_train'].items():
    print(f'  {cls!r:>6}: {frac:.3%}')


## 3. EDA (somente no treino)

EDA mínima para o checkpoint: distribuição de classes, missingness, distribuições de variáveis-chave. Será aprofundada no relatório.

In [ ]:
# Distribuição da classe alvo
fig, ax = plt.subplots(figsize=(7, 3.5))
ord_classes = ['<30', '>30', 'NO']
counts = y_train.value_counts().reindex(ord_classes)
counts.plot(kind='bar', ax=ax, color=['#d62728', '#ff7f0e', '#2ca02c'])
ax.set_title('Distribuição de readmitted (treino)')
ax.set_xlabel('Classe')
ax.set_ylabel('Quantidade')
for i, v in enumerate(counts):
    ax.text(i, v, f'{v:,}\n({v/counts.sum():.1%})', ha='center', va='bottom')
ax.set_ylim(0, counts.max() * 1.18)
fig.tight_layout()
fig.savefig('reports/figures/eda_class_balance.png', dpi=120)
plt.show()


In [ ]:
# Missingness ('?' marca ausência no UCI Diabetes)
X_eda = X_train.replace('?', np.nan)
miss = X_eda.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(9, max(3, 0.35 * len(miss))))
miss.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Fração de valores ausentes')
ax.set_title('Missingness por coluna (treino)')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig('reports/figures/eda_missingness.png', dpi=120)
plt.show()
print(f'{len(miss)} colunas com algum valor ausente.')


In [ ]:
# Distribuição de algumas variáveis numéricas-chave por classe
numeric_show = ['time_in_hospital', 'num_medications', 'number_inpatient', 'number_diagnoses']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), numeric_show):
    for cls, color in zip(ord_classes, ['#d62728', '#ff7f0e', '#2ca02c']):
        mask = y_train == cls
        ax.hist(X_train.loc[mask, col], bins=30, alpha=0.45, label=cls, color=color)
    ax.set_title(col)
    ax.set_yscale('log')
    ax.legend(title='readmitted', fontsize=8)
fig.suptitle('Distribuições por classe (escala log)')
fig.tight_layout()
fig.savefig('reports/figures/eda_numeric_by_class.png', dpi=120)
plt.show()


## 4. Pipeline baseline (pré-processamento mínimo)

Baseline conforme exigência Seção 7.4: limpeza estrutural + imputação + OHE + StandardScaler. **Sem** balanceamento, **sem** engenharia de características. Esses elementos entram nas variantes (Seção 6).

Tudo está num `Pipeline` sklearn — o `fit` só acontece dentro de cada fold da CV (via `GridSearchCV` / `RandomizedSearchCV`), nunca no dataset inteiro.

In [ ]:
preproc = build_baseline_pipeline()
preproc_fit_demo = build_baseline_pipeline().fit(X_train.sample(2000, random_state=0))
n_features_out = preproc_fit_demo.transform(X_train.head(5)).shape[1]
print('Pipeline baseline:')
print(preproc)
print(f'\nApós OHE: {n_features_out} features (em amostra de demonstração).')


## 5. Busca de hiperparâmetros (baseline)

Para o **checkpoint (20/05/2026)** rodamos 3 modelos: Decision Tree, Random Forest e LightGBM. Cobertura dos 10 algoritmos será fechada antes da apresentação (27/05). Os outros 7 já estão registrados em `src/models.py` (flag `enabled_for_checkpoint=False`).

Cada busca:
- usa **5-fold estratificado** com `random_state=42` (`src/utils.get_cv()`)
- otimiza `f1_macro`
- salva `cv_results_` em `reports/cv_results/<modelo>__baseline.csv`
- plota **curva treino vs validação** em `reports/figures/<modelo>__baseline__train_vs_val.png`

In [ ]:
searches: dict[str, object] = {}
summaries: list[dict] = []
fold_scores: dict[str, list[float]] = {}

for spec in checkpoint_models():
    print(f'\n=== {spec.name} ({spec.search}, {spec.notes}) ===')
    s = run_search(
        spec=spec,
        X=X_train,
        y=y_train,
        prep_factory=build_baseline_pipeline,
        tag='baseline',
    )
    searches[spec.name] = s
    summary = summary_from_search(s, label=spec.name)
    summaries.append(summary)
    fold_scores[spec.name] = best_per_fold_scores(s)
    print(format_summary(summary))
    print(f'best_params: {summary["best_params"]}')


In [ ]:
# Resumo consolidado dos modelos do checkpoint
df_summary = pd.DataFrame([
    {
        'modelo': s['label'],
        'f1_macro_val': s['f1_macro_mean'],
        'f1_macro_val_std': s['f1_macro_std'],
        'f1_macro_treino': s['f1_macro_train_mean'],
        'gap_treino_val': s['f1_macro_train_mean'] - s['f1_macro_mean'],
    }
    for s in summaries
]).sort_values('f1_macro_val', ascending=False).reset_index(drop=True)
df_summary


In [ ]:
# Comparação pareada (Wilcoxon + paired t-test) entre os modelos do checkpoint.
# Decisão de variante vencedora exige isto (pré-requisito da rubrica).
names = list(fold_scores.keys())
pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        r = paired_compare(fold_scores[a], fold_scores[b])
        pairs.append({
            'A': a, 'B': b,
            'F1_A': r.mean_a, 'F1_B': r.mean_b,
            'diff_B_minus_A': r.diff,
            'wilcoxon_p': r.wilcoxon_p,
            'paired_t_p': r.t_test_p,
        })
pd.DataFrame(pairs)


## 6. Variantes do treino + nova busca HP

**A ser preenchido após o checkpoint.** Cada variante deve ser um novo `build_*_pipeline()` em `src/preprocessing.py`, reaproveitando blocos do baseline:

- **Balanceamento**: SMOTE / undersampling / `class_weight` (use `imblearn.Pipeline` quando precisar resamplear dentro do CV).
- **Escalas**: trocar `StandardScaler` por `RobustScaler` / `MinMaxScaler`.
- **Codificações**: `TargetEncoder` em colunas de alta cardinalidade (`payer_code`, `medical_specialty`); manter OHE para o resto.
- **Engenharia de características**: derivar contagens, agrupamentos ICD-9 mais finos, etc.

Para cada variante, rodar a mesma `run_search` com `tag='variante_X'` e os mesmos 10 modelos. Os `fold_scores` ficam pareados (mesmos folds) por construção.

In [ ]:
# Placeholder — exemplo de como uma variante seria invocada:
#
# from src.preprocessing import build_smote_pipeline  # ainda a criar
# for spec in [m for m in MODEL_REGISTRY.values()]:
#     run_search(spec, X_train, y_train, build_smote_pipeline, tag='smote')
print('Seção 6: a preencher após o checkpoint.')


## 7. Comparação baseline vs variantes (Wilcoxon / paired t-test)

Para cada modelo, comparar `fold_scores[modelo]` do baseline com `fold_scores[modelo]` de cada variante usando `paired_compare()`. Tabela final com:
- F1-macro médio por config
- diferença vs baseline
- p-valor Wilcoxon e paired t-test
- decisão (manter baseline / promover variante)

In [ ]:
print('Seção 7: a preencher após o checkpoint.')


## 8. Seleção do melhor modelo

Justificar a escolha com:
- métrica F1-macro com desvio entre folds
- comparação pareada vs baseline (p-valor)
- curva treino vs validação da configuração escolhida
- considerações de custo computacional e interpretabilidade (alinhadas ao briefing)

In [ ]:
print('Seção 8: a preencher após o checkpoint.')


## 9. Avaliação final no teste (única seção que toca o teste)

**Esta é a única seção que pode chamar `load_test`.** Não copie esta célula para outras seções.

In [ ]:
# from src.data_loader import load_test
# from src.evaluation import plot_confusion, plot_roc_pr_macro, full_report
#
# X_test, y_test = load_test(unlock_token='I_AM_IN_FINAL_EVALUATION')
# best_model = searches['<modelo_vencedor>'].best_estimator_  # já refitado em todo X_train
# y_pred = best_model.predict(X_test)
# y_score = best_model.predict_proba(X_test)
#
# plot_confusion(y_test, y_pred, classes=['<30','>30','NO'],
#                title='Avaliação final — matriz normalizada',
#                filename='final_confusion.png')
# plot_roc_pr_macro(y_test, y_score, classes=['<30','>30','NO'],
#                   title_prefix='Final', filename_prefix='final')
# print(full_report(y_test, y_pred, classes=['<30','>30','NO']))
print('Seção 9: descomentar e rodar SOMENTE após selecionar o modelo final.')


## 10. Deployment, riscos e monitoramento

**A ser preenchido com:**
- Cenário de uso: integração com o sistema do hospital (acionamento pós-alta).
- Riscos: viés em grupos sub-representados, custos de falso negativo na classe `<30`.
- Plano de monitoramento: drift de features, F1-macro mensal estratificado por unidade.
- Retreinamento: condições de gatilho (queda >X% no F1-macro de produção).

In [ ]:
print('Seção 10: documentação no relatório final.')
